In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
file_path = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2020/2020-01-21/spotify_songs.csv"

df = pd.read_csv(file_path)
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [3]:
print("original dataset's shape" , df.shape)

original dataset's shape (32833, 23)


## Data cleaning

In [4]:
#####
# time processing

# Duration in minutes is easier to read than raw milliseconds
df['duration_min'] = df['duration_ms'] / 60000
df["duration_min"].describe()

count    32833.000000
mean         3.763330
std          0.997233
min          0.066667
25%          3.130317
50%          3.600000
75%          4.226417
max          8.630167
Name: duration_min, dtype: float64

In [5]:
#####
# year processing

s = df["track_album_release_date"].astype(str).str.strip()

# detect year-only values like "2020"
year_only = s.str.fullmatch(r"\d{4}")

# prepare output column (nullable integer)
df["release_year"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

# 1️⃣ year-only values → use directly
df.loc[year_only, "release_year"] = s[year_only].astype(int)

# 2️⃣ all other date formats → parse then extract year
parsed = pd.to_datetime(
    s[~year_only],
    dayfirst=False,
    errors="coerce"
)

df.loc[~year_only, "release_year"] = parsed.dt.year
df["release_year"].describe()

count        32802.0
mean     2011.173587
std        11.360195
min           1957.0
25%           2008.0
50%           2016.0
75%           2019.0
max           2020.0
Name: release_year, dtype: Float64

In [6]:
df.isna().sum()[df.isna().sum() > 0]

track_name           5
track_artist         5
track_album_name     5
release_year        31
dtype: int64

In [7]:
df = df.dropna()

In [8]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [9]:
df = df[df["release_year"] >= 1970]

In [10]:
print("cleaned dataset's shape" , df.shape)
print("=" * 55)
print(df.describe())

cleaned dataset's shape (32631, 25)
       track_popularity  danceability        energy           key  \
count      32631.000000  32631.000000  32631.000000  32631.000000   
mean          42.448745      0.655681      0.699093      5.378842   
std           24.975076      0.144711      0.180651      3.611898   
min            0.000000      0.000000      0.000175      0.000000   
25%           24.000000      0.564000      0.582000      2.000000   
50%           45.000000      0.672000      0.722000      6.000000   
75%           62.000000      0.761000      0.840000      9.000000   
max          100.000000      0.983000      1.000000     11.000000   

           loudness          mode   speechiness  acousticness  \
count  32631.000000  32631.000000  32631.000000  32631.000000   
mean      -6.701845      0.564525      0.107368      0.174487   
std        2.979180      0.495827      0.101466      0.219107   
min      -46.448000      0.000000      0.000000      0.000000   
25%       -8.1450

In [11]:
df.to_csv('cleaned_spotify.csv', index=False)

# Creating dataset for step 3

In [12]:
# ========== PREPROCESSING FOR WEBSITE JSONS ==========
import json
import pathlib
import numpy as np

# 1. Normalise tempo to [0,1] for radar chart
t_min, t_max = df["tempo"].min(), df["tempo"].max()
df["tempo_norm"] = (df["tempo"] - t_min) / (t_max - t_min)

# 2. Tag tracks that appear in more than one playlist genre
track_genre_count = df.groupby("track_id")["playlist_genre"].nunique()
df["genre_count"] = df["track_id"].map(track_genre_count)
df["genre_group"] = np.where(df["genre_count"] > 1, "multi-genre", "single-genre")

# Create a version without multi‑genre tracks (each song belongs to one genre)
single_df = df[df["genre_group"] == "single-genre"].copy()
print(f"Rows in single_df (no multi‑genre): {len(single_df):,}")

# 3. Define features and display names
RADAR_FEATURES = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "tempo_norm",
    "speechiness",
]

DISPLAY_NAMES = {
    "danceability": "Danceability",
    "energy":       "Energy",
    "valence":      "Valence",
    "acousticness": "Acousticness",
    "tempo_norm":   "Tempo",
    "speechiness":  "Speechiness",
}

GENRE_COLORS = {
    "edm":   "#1DB954",
    "latin": "#F72585",
    "pop":   "#4CC9F0",
    "r&b":   "#7B2D8B",
    "rap":   "#F77F00",
    "rock":  "#D62828",
}

# 4. Output directory (relative to notebook location)
OUT_DIR = pathlib.Path("../specs/src/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Export 1: genre_profiles.json ----------
genre_mean = (
    single_df
    .groupby("playlist_genre")[RADAR_FEATURES]
    .mean()
    .round(4)
)

genre_profiles = []
for genre in genre_mean.index:
    row = genre_mean.loc[genre]
    genre_profiles.append({
        "genre": genre,
        "color": GENRE_COLORS.get(genre, "#999999"),
        "features": {
            DISPLAY_NAMES[feat]: float(row[feat])
            for feat in RADAR_FEATURES
        },
    })

out_path = OUT_DIR / "genre_profiles.json"
with open(out_path, "w") as f:
    json.dump(genre_profiles, f, indent=2)
print(f"Saved: {out_path}")

# # ---------- Export 2: top_songs.json ----------
# top_songs_list = []
# for genre in single_df["playlist_genre"].unique():
#     genre_tracks = (
#         single_df[single_df["playlist_genre"] == genre]
#         .drop_duplicates(subset="track_id")
#         .sort_values("track_popularity", ascending=False)
#         .head(3)
#     )
#     for _, row in genre_tracks.iterrows():
#         top_songs_list.append({
#             "genre":        genre,
#             "track_name":   row["track_name"],
#             "track_artist": row["track_artist"],
#             "popularity":   int(row["track_popularity"]),
#             "danceability": round(float(row["danceability"]), 3),
#             "energy":       round(float(row["energy"]), 3),
#             "valence":      round(float(row["valence"]), 3),
#             "tempo":        round(float(row["tempo"]), 1),
#             "speechiness":  round(float(row["speechiness"]), 3),
#             "acousticness": round(float(row["acousticness"]), 3),
#         })

# out_path = OUT_DIR / "top_songs.json"
# with open(out_path, "w") as f:
#     json.dump(top_songs_list, f, indent=2)
# print(f"Saved: {out_path} (entries: {len(top_songs_list)})")

# # ---------- Export 3: genre_yearly.json ----------
# TREND_FEATURES = ["danceability", "energy", "valence", "speechiness"]

# yearly_rows = []
# for genre in single_df["playlist_genre"].unique():
#     sub = single_df[single_df["playlist_genre"] == genre]
#     yearly = (
#         sub.groupby("release_year")[TREND_FEATURES]
#         .mean()
#         .round(4)
#         .reset_index()
#     )
#     for _, row in yearly.iterrows():
#         entry = {"genre": genre, "year": int(row["release_year"])}
#         for feat in TREND_FEATURES:
#             entry[feat] = float(row[feat])
#         yearly_rows.append(entry)

# out_path = OUT_DIR / "genre_yearly.json"
# with open(out_path, "w") as f:
#     json.dump(yearly_rows, f, indent=2)
# print(f"Saved: {out_path}")

print("\n✅ All JSON files written to spec/src/data/")

Rows in single_df (no multi‑genre): 28,127
Saved: ../specs/src/data/genre_profiles.json

✅ All JSON files written to spec/src/data/


In [13]:
import pandas as pd

# Load the data
df = pd.read_csv("cleaned_spotify.csv")

# Choose the genre column (commonly used)
genre_col = "playlist_genre"

# Select audio feature columns
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

# Group by genre and calculate mean values
avg_features_by_genre = (
    df.groupby(genre_col)[audio_features]
      .mean()
      .reset_index()
)

# Display result
print(avg_features_by_genre)

# (Optional) Save to CSV
avg_features_by_genre.to_csv("avg_audio_features_by_genre.csv", index=False)

  playlist_genre  danceability    energy  loudness  speechiness  acousticness  \
0            edm      0.655041  0.802476 -5.427445     0.086695      0.081504   
1          latin      0.713321  0.708306 -6.263361     0.102665      0.210902   
2            pop      0.639312  0.701049 -6.314039     0.074012      0.170770   
3            r&b      0.670944  0.591207 -7.852773     0.117126      0.258798   
4            rap      0.718507  0.650741 -7.040576     0.197581      0.192501   
5           rock      0.520875  0.736073 -7.522926     0.057767      0.140262   

   instrumentalness  liveness   valence       tempo  
0          0.218578  0.211859  0.400656  125.768024  
1          0.044472  0.180669  0.605465  118.633667  
2          0.059817  0.176867  0.503407  120.730075  
3          0.028686  0.175026  0.530789  114.147148  
4          0.076068  0.191639  0.504877  120.666710  
5          0.061399  0.203139  0.534365  125.113979  


In [15]:
import pandas as pd
import json
import numpy as np

# ------------------------------
# 1. Load and process CSV
# ------------------------------
df = pd.read_csv("cleaned_spotify.csv")

# Audio feature columns in CSV (same as in your code)
audio_features_csv = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "valence",
    "tempo"
]

# Normalise tempo to 0-1 range (global max)
max_tempo = df["tempo"].max()
df["tempo_normalised"] = df["tempo"] / max_tempo

# Group by genre and compute mean
genre_col = "playlist_genre"
grouped = df.groupby(genre_col)

# Build a dictionary with the means for each feature
csv_means = {}
for genre, grp in grouped:
    csv_means[genre] = {
        "danceability": grp["danceability"].mean(),
        "energy": grp["energy"].mean(),
        "speechiness": grp["speechiness"].mean(),
        "acousticness": grp["acousticness"].mean(),
        "valence": grp["valence"].mean(),
        "tempo": grp["tempo_normalised"].mean()
    }

# ------------------------------
# 2. Load JSON file
# ------------------------------
with open("../specs/src/data/genre_profiles.json", "r") as f:
    json_data = json.load(f)

# Convert JSON to the same structure: key = genre, value = dict of features
json_means = {}
for entry in json_data:
    genre = entry["genre"]
    # JSON uses "Tempo" (capital T) and other features with capital first letter
    json_means[genre] = {
        "danceability": entry["features"]["Danceability"],
        "energy": entry["features"]["Energy"],
        "speechiness": entry["features"]["Speechiness"],
        "acousticness": entry["features"]["Acousticness"],
        "valence": entry["features"]["Valence"],
        "tempo": entry["features"]["Tempo"]
    }

# ------------------------------
# 3. Compare and print
# ------------------------------
print("Checking genre profiles against cleaned_spotify.csv\n")
print(f"{'Genre':<8} {'Feature':<15} {'CSV mean':<10} {'JSON value':<10} {'Difference':<10}")
print("-" * 60)

for genre in sorted(csv_means.keys()):
    for feature in ["danceability", "energy", "speechiness", "acousticness", "valence", "tempo"]:
        csv_val = csv_means[genre][feature]
        json_val = json_means[genre][feature]
        diff = csv_val - json_val
        print(f"{genre:<8} {feature:<15} {csv_val:<10.4f} {json_val:<10.4f} {diff:<10.4f}")

# Also report any missing genres or features
csv_genres = set(csv_means.keys())
json_genres = set(json_means.keys())
print("\nGenres in CSV only:", csv_genres - json_genres)
print("Genres in JSON only:", json_genres - csv_genres)

Checking genre profiles against cleaned_spotify.csv

Genre    Feature         CSV mean   JSON value Difference
------------------------------------------------------------
edm      danceability    0.6550     0.6549     0.0001    
edm      energy          0.8025     0.8132     -0.0107   
edm      speechiness     0.0867     0.0874     -0.0007   
edm      acousticness    0.0815     0.0734     0.0081    
edm      valence         0.4007     0.3916     0.0091    
edm      tempo           0.5253     0.5281     -0.0028   
latin    danceability    0.7133     0.7144     -0.0011   
latin    energy          0.7083     0.7142     -0.0059   
latin    speechiness     0.1027     0.1014     0.0013    
latin    acousticness    0.2109     0.2128     -0.0019   
latin    valence         0.6055     0.6177     -0.0122   
latin    tempo           0.4955     0.4960     -0.0005   
pop      danceability    0.6393     0.6369     0.0024    
pop      energy          0.7010     0.7026     -0.0016   
pop      speechi